<a href="https://colab.research.google.com/github/hamagami/ROUTE2026/blob/main/medical_ai_cnn_anomaly_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 医用画像AI実習：CNNを用いた異常検知システム

このノートブックでは、公開医用画像データセット **PneumoniaMNIST** を用いて、CNN（Convolutional Neural Network）による異常検知システムを作成します。

前回のAutoencoder版では、**正常画像だけを学習し、再構成誤差の大きさで異常を検出**しました。

今回は、CNNを用いて **正常 / 肺炎** を直接分類し、CNNの出力する「肺炎である確率」を **異常スコア** として扱います。

## 学習目標

- Google ColabとGoogle Driveを用いた実習環境を構築する
- 医用画像データセットをPythonで読み込む
- PyTorchでCNNを定義する
- CNNを学習・評価する
- 出力確率を異常スコアとして扱う
- ROC-AUC、混同行列、F1スコアで異常検知性能を評価する
- Grad-CAMにより、CNNが画像のどこを見て判断しているかを可視化する

## 注意

この教材は教育目的です。ここで作成するモデルは臨床診断に使用できるものではありません。


## 第1回：ColabとGoogle Driveの準備
Google Driveをマウントし、学習済みモデルや実験結果を保存できるようにします。Colabではランタイムが切れるとファイルが消えるため、永続保存にはDriveを使います。

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/medical_ai_cnn_anomaly')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('保存先:', PROJECT_DIR)


## 第2回：ライブラリのインストール
MedMNISTは、医用画像データセットをMNISTのように扱いやすくしたパッケージです。今回は胸部X線画像のPneumoniaMNISTを使用します。

In [ ]:
!pip -q install medmnist scikit-learn matplotlib tqdm


## 第3回：Python / PyTorch / GPUの確認
Colabのメニューから **ランタイム → ランタイムのタイプを変更 → GPU** を選ぶと、学習が速くなります。

In [ ]:
import os
import random
import numpy as np
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用デバイス:', DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 第4回：PneumoniaMNISTの読み込み
PneumoniaMNISTは、胸部X線画像を **正常 / 肺炎** に分類するデータセットです。ここではPyTorchのDatasetとして読み込みます。

In [ ]:
import medmnist
from medmnist import INFO
from torchvision import transforms
from torch.utils.data import DataLoader

DATA_FLAG = 'pneumoniamnist'
info = INFO[DATA_FLAG]
print(info)
DataClass = getattr(medmnist, info['python_class'])

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = DataClass(split='train', transform=transform, download=True)
val_dataset   = DataClass(split='val',   transform=transform, download=True)
test_dataset  = DataClass(split='test',  transform=transform, download=True)

print('train:', len(train_dataset))
print('val  :', len(val_dataset))
print('test :', len(test_dataset))
print('classes:', info['label'])


## 第5回：画像とラベルの確認
まずは画像を表示し、データがどのような形になっているか確認します。

In [ ]:
import matplotlib.pyplot as plt
label_names = info['label']

def show_samples(dataset, n=12):
    plt.figure(figsize=(12, 4))
    for i in range(n):
        img, label = dataset[i]
        label_int = int(label[0])
        plt.subplot(2, n//2, i+1)
        plt.imshow(img.squeeze(), cmap='gray')
        plt.title(f'{label_int}: {label_names[str(label_int)]}')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, n=12)


## 第6回：ラベル分布の確認
医用画像では、正常と異常の枚数が偏っていることがよくあります。この偏りは、Accuracyだけを見てはいけない理由にもなります。

In [ ]:
from collections import Counter
for name, ds in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    labels = [int(ds[i][1][0]) for i in range(len(ds))]
    print(name, Counter(labels))


## 第7回：DataLoaderの作成
DataLoaderは、データをミニバッチに分けて取り出すための仕組みです。CNNの学習では、画像を1枚ずつではなく、複数枚まとめて処理します。

In [ ]:
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print('画像バッチ:', images.shape)
print('ラベルバッチ:', labels.shape)


## 第8回：CNNモデルの定義
CNNは、画像の局所的なパターンを畳み込み層で抽出し、最終的に正常か異常かを判定します。ここでは、教育用に小さめのCNNを作ります。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        x = self.features(x)
        logits = self.classifier(x)
        return logits.squeeze(1)

model = SimpleCNN().to(DEVICE)
print(model)


## 第9回：損失関数と最適化手法
2値分類では `BCEWithLogitsLoss` を使います。この損失関数は、シグモイド関数と2値交差エントロピーを内部でまとめて計算します。PneumoniaMNISTはラベルの偏りがあるため、陽性クラスに重みを付けます。

In [ ]:
train_labels = np.array([int(train_dataset[i][1][0]) for i in range(len(train_dataset))])
num_negative = np.sum(train_labels == 0)
num_positive = np.sum(train_labels == 1)
pos_weight_value = num_negative / num_positive
print('negative:', num_negative)
print('positive:', num_positive)
print('pos_weight:', pos_weight_value)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, dtype=torch.float32).to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


## 第10回：学習・検証関数の作成
1エポック分の学習と、検証データでの評価を行う関数を作ります。この部分は、PyTorchによるAI開発の基本パターンです。

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE)
        labels = labels.float().view(-1).to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.float().view(-1).to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.sigmoid(logits)
        total_loss += loss.item() * images.size(0)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)
    acc = accuracy_score(all_labels, preds)
    auc_value = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader.dataset), acc, auc_value, all_labels, all_probs


## 第11回：CNNの学習
検証データのAUCが最も高いモデルをGoogle Driveに保存します。

In [ ]:
EPOCHS = 10
best_auc = 0.0
history = []
model_path = PROJECT_DIR / 'best_simple_cnn_pneumoniamnist.pt'

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_auc, _, _ = evaluate(model, val_loader, criterion)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc, 'val_auc': val_auc})
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_auc={val_auc:.4f}')
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), model_path)
        print('  -> best model saved')
print('best val AUC:', best_auc)
print('model saved to:', model_path)


## 第12回：学習曲線の確認
LossやAUCの推移を見ることで、学習が進んでいるか、過学習していないかを確認します。

In [ ]:
import pandas as pd
hist_df = pd.DataFrame(history)
display(hist_df)

plt.figure(figsize=(6,4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], marker='o', label='train loss')
plt.plot(hist_df['epoch'], hist_df['val_loss'], marker='o', label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist_df['epoch'], hist_df['val_auc'], marker='o', label='val AUC')
plt.xlabel('epoch')
plt.ylabel('AUC')
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()


## 第13回：テストデータでの最終評価
保存した最良モデルを読み込み、テストデータで評価します。ここでCNNの出力確率を **異常スコア** として扱います。

- 0に近い：正常らしい
- 1に近い：肺炎、つまり異常らしい

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, f1_score
best_model = SimpleCNN().to(DEVICE)
best_model.load_state_dict(torch.load(model_path, map_location=DEVICE))

test_loss, test_acc, test_auc, y_true, anomaly_scores = evaluate(best_model, test_loader, criterion)
print('test loss:', test_loss)
print('test accuracy:', test_acc)
print('test ROC-AUC:', test_auc)


## 第14回：ROC曲線としきい値の考え方
異常検知では、出力確率そのものよりも、どのしきい値で異常と判断するかが重要です。しきい値を低くすると異常を見逃しにくくなりますが、正常を異常と誤判定しやすくなります。

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, anomaly_scores)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f'ROC curve AUC={roc_auc:.3f}')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()


## 第15回：F1スコアが最大になるしきい値を探す
単純に0.5で判定するのではなく、検証データを使って良いしきい値を決めます。ここではF1スコアが最大になるしきい値を選びます。

In [ ]:
_, _, _, y_val, val_scores = evaluate(best_model, val_loader, criterion)
candidate_thresholds = np.linspace(0.01, 0.99, 99)
best_threshold = 0.5
best_f1 = 0.0
for th in candidate_thresholds:
    preds = (val_scores >= th).astype(int)
    f1 = f1_score(y_val, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = th
print('best threshold on validation:', best_threshold)
print('best validation F1:', best_f1)
y_pred = (anomaly_scores >= best_threshold).astype(int)
print(classification_report(y_true, y_pred, target_names=['normal', 'pneumonia']))


## 第16回：混同行列の確認
混同行列を見ると、どの種類の誤りが多いかがわかります。

- True Negative：正常を正常と判定
- False Positive：正常を異常と誤判定
- False Negative：異常を正常と誤判定
- True Positive：異常を異常と判定

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print(cm)
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks([0,1], ['normal', 'pneumonia'])
plt.yticks([0,1], ['normal', 'pneumonia'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center')
plt.colorbar()
plt.show()


## 第17回：異常スコアの分布
正常画像と肺炎画像で、CNNの異常スコアがどのように分布しているかを確認します。

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(anomaly_scores[y_true == 0], bins=30, alpha=0.6, label='normal')
plt.hist(anomaly_scores[y_true == 1], bins=30, alpha=0.6, label='pneumonia')
plt.axvline(best_threshold, linestyle='--', label=f'threshold={best_threshold:.2f}')
plt.xlabel('anomaly score = P(pneumonia)')
plt.ylabel('count')
plt.legend()
plt.grid(True)
plt.show()


## 第18回：正しく判定した画像・誤判定した画像を見る
AIの評価では、数値だけでなく、実際にどの画像を間違えたのか確認することが重要です。

In [ ]:
all_test_images = []
all_test_labels = []
for img, label in test_dataset:
    all_test_images.append(img)
    all_test_labels.append(int(label[0]))
all_test_images = torch.stack(all_test_images)
all_test_labels = np.array(all_test_labels)

def show_cases(indices, title, n=12):
    indices = list(indices)[:n]
    if len(indices) == 0:
        print(title, ': 該当なし')
        return
    plt.figure(figsize=(12, 4))
    for k, idx in enumerate(indices):
        img = all_test_images[idx].squeeze().numpy()
        true_label = all_test_labels[idx]
        score = anomaly_scores[idx]
        pred = int(score >= best_threshold)
        plt.subplot(2, n//2, k+1)
        plt.imshow(img, cmap='gray')
        plt.title(f'T:{true_label} P:{pred}
score={score:.2f}')
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

true_positive = np.where((y_true == 1) & (y_pred == 1))[0]
false_negative = np.where((y_true == 1) & (y_pred == 0))[0]
false_positive = np.where((y_true == 0) & (y_pred == 1))[0]
true_negative = np.where((y_true == 0) & (y_pred == 0))[0]
show_cases(true_positive, 'True Positive: 異常を異常と判定')
show_cases(false_negative, 'False Negative: 異常を正常と誤判定')
show_cases(false_positive, 'False Positive: 正常を異常と誤判定')
show_cases(true_negative, 'True Negative: 正常を正常と判定')


## 第19回：Grad-CAMによる判断根拠の可視化
CNNが画像のどの領域を見て「肺炎らしい」と判断しているかを、Grad-CAMで可視化します。Grad-CAMは完全な説明ではありませんが、医用画像AIでは、モデルがどこに反応しているか確認するための基本的な方法です。

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    def __call__(self, x):
        self.model.zero_grad()
        logits = self.model(x)
        score = logits.squeeze()
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    def close(self):
        self.forward_hook.remove()
        self.backward_hook.remove()


In [ ]:
cam_extractor = GradCAM(best_model, best_model.features[8])
high_score_indices = np.argsort(-anomaly_scores)[:8]
plt.figure(figsize=(12, 6))
for k, idx in enumerate(high_score_indices):
    x = all_test_images[idx:idx+1].to(DEVICE)
    cam = cam_extractor(x)
    img = all_test_images[idx].squeeze().numpy()
    plt.subplot(2, 4, k+1)
    plt.imshow(img, cmap='gray')
    plt.imshow(cam, alpha=0.45)
    plt.title(f'T:{all_test_labels[idx]} score={anomaly_scores[idx]:.2f}')
    plt.axis('off')
plt.suptitle('Grad-CAM: CNNが反応した領域')
plt.tight_layout()
plt.show()
cam_extractor.close()


## 第20回：モデルの再読み込みと1枚画像の判定関数
実習の最後に、保存済みモデルを読み込んで1枚の画像を判定する関数を作ります。これは、簡単なAIアプリケーション化への第一歩です。

In [ ]:
def load_cnn_model(model_path):
    model = SimpleCNN().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()
    return model

@torch.no_grad()
def predict_one_image(model, image_tensor, threshold=0.5):
    # image_tensor: shape [1, 28, 28]
    x = image_tensor.unsqueeze(0).to(DEVICE)
    logit = model(x)
    score = torch.sigmoid(logit).item()
    pred = int(score >= threshold)
    return {
        'anomaly_score': score,
        'prediction': pred,
        'prediction_name': 'pneumonia/anomaly' if pred == 1 else 'normal'
    }

loaded_model = load_cnn_model(model_path)
sample_img, sample_label = test_dataset[0]
result = predict_one_image(loaded_model, sample_img, threshold=best_threshold)
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f"true={int(sample_label[0])}, score={result['anomaly_score']:.3f}, pred={result['prediction_name']}")
plt.axis('off')
plt.show()
print(result)


# まとめ：CNN版異常検知で学んだこと

この実習では、CNNを用いて胸部X線画像の正常 / 肺炎分類を行い、CNNの出力確率を異常スコアとして利用しました。

## Autoencoder版との違い

| 観点 | Autoencoder版 | CNN版 |
|---|---|---|
| 学習に使うラベル | 基本的に正常画像のみ | 正常 / 異常ラベルを使用 |
| 異常スコア | 再構成誤差 | 異常クラスの出力確率 |
| 得意な状況 | 異常画像が少ない場合 | 正常・異常ラベルが十分ある場合 |
| 弱点 | 再構成誤差が臨床的異常と一致しないことがある | 学習済みの異常タイプに依存しやすい |
| 説明性 | 誤差画像を見られる | Grad-CAMなどで確認できる |

## 発展課題

1. CNNの層を増やすと性能は上がるか
2. DropoutやBatchNormを外すとどうなるか
3. しきい値をF1最大ではなく、感度重視で決めるとどうなるか
4. Autoencoder版とCNN版で、同じ画像に対する異常スコアを比較する
5. Grad-CAMが本当に病変部位を見ているかを考察する
6. EfficientNetやResNetなどの事前学習モデルを使う
7. 28×28ではなく、224×224に拡大して学習する
